In [2]:
from dash import Dash, html
import dash_ag_grid as dag
import pandas as pd


# initialize app
app = Dash()

# Requires Dash 2.17.0 or later
app.layout = [html.Div(children='Hello World')]

if __name__ == '__main__':
    app.run(debug=True)

In [3]:
# Import packages
from dash import Dash, html, dcc, callback, Output, Input
import dash_ag_grid as dag
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import plotnine as p9
import geopandas as gpd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import plotly.io as pio
pio.renderers.default = "notebook"

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import time

# Incorporate data
file_paths = ["WorldCupMatches.csv", "WorldCupPlayers.csv", "WorldCups.csv"]
df_list = []
for i in file_paths:
# Set the path to the file you'd like to load
    file_path = i

    # Load the latest version
    df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "abecklas/fifa-world-cup",
    file_path,
    # Provide any additional arguments like 
    # sql_query or pandas_kwargs. See the 
    # documenation for more information:
    # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
    )
    df_list.append(df)

matches_df = df_list[0]
players_df = df_list[1]
cups_df = df_list[2]

# Home / away win rates by stadium
m = matches_df.copy()
m['Stadium'] = m['Stadium'].astype(str).str.strip()
m['City'] = m['City'].astype(str).str.strip()
m['Home Team Goals'] = pd.to_numeric(m['Home Team Goals'], errors='coerce')
m['Away Team Goals'] = pd.to_numeric(m['Away Team Goals'], errors='coerce')
m = m.dropna(subset=['Stadium', 'Home Team Goals', 'Away Team Goals'])
m = m[m['Stadium'].ne('') & m['Stadium'].ne('nan')]

m['home_win'] = (m['Home Team Goals'] > m['Away Team Goals']).astype(int)
m['away_win'] = (m['Home Team Goals'] < m['Away Team Goals']).astype(int)
m['draw'] = (m['Home Team Goals'] == m['Away Team Goals']).astype(int)

stadium_rates = (
    m.groupby('Stadium', as_index=False)
     .agg(
         City=('City', 'first'),
         matches=('home_win', 'size'),
         home_win_rate=('home_win', 'mean'),
         away_win_rate=('away_win', 'mean'),
         draw_rate=('draw', 'mean'),
     )
     .sort_values('matches', ascending=False)
)

stadium_rates['home_win_rate'] *= 100
stadium_rates['away_win_rate'] *= 100
stadium_rates['draw_rate'] *= 100

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div(children='My First App with Data, Graph, and Controls'),
    html.Hr(),
    dcc.RadioItems(options=['home_win_rate', 'away_win_rate'], value='Stadium', id='controls-and-radio-item'),
    dag.AgGrid(
        rowData=m.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    ),
    dcc.Graph(figure={}, id='controls-and-graph')
]

# Add controls to build the interaction
@callback(
    Output(component_id='controls-and-graph', component_property='figure'),
    Input(component_id='controls-and-radio-item', component_property='value')
)
def update_graph(col_chosen):
    fig = px.histogram(df, x='continent', y=col_chosen, histfunc='avg')
    return fig

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

/Users/coraalbers/anaconda3/envs/capstone/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
/var/folders/60/m3wm021x08vbp7kk3s0t94c40000gn/T/ipykernel_68115/3459893761.py:32: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
